In [ ]:
from amplpy import AMPL
import itertools

# Generate vectors as per your previous function
def generate_valid_vectors(N):
    """Generate all 0/1 vectors with the constraint: if vec[i] == 1, then vec[N+i] must be 1"""
    for vec in itertools.product([0, 1], repeat=2*N):
        if all(not (vec[i] == 1 and vec[N+i] == 0) for i in range(N)):
            yield vec
# Initialize AMPL session
ampl = AMPL()
ampl.read("/content/your_model.mod")
# Optional: read data
# ampl.readData("/content/your_data.dat")

# Since all variables are integer, take all of them
int_vars = list(ampl.getVariables())
n = len(int_vars)



# Close this session if needed; we'll create new ones in iterations
ampl.close()


In [ ]:
def solve_relaxations_with_value_store(model_path, data_path, bounds_vectors,
                                       int_var_names, solver="cbc", tol=1e-6,
                                       max_iters=20, state_round=4):
    """
    Iteratively solve LP relaxations and update recursive V(s).
    int_var_names: list of names of integer variables (extracted beforehand)
    """
    def normalize_state(bounds):
        return tuple(round(float(b), state_round) for b in bounds)

    V_store = {}
    all_results = []

    for start_idx, start_bounds in enumerate(bounds_vectors, start=1):
        print(f"\n=== Starting initial vector {start_idx} ===")

        ampl = AMPL()
        ampl.setOption("solver", solver)
        ampl.read(model_path)
        if data_path:
            ampl.readData(data_path)

        # Retrieve variables by name
        int_vars = [ampl.getVariable(name) for name in int_var_names]
        n = len(int_vars)

        # Initial state
        s = normalize_state(start_bounds)
        if s not in V_store:
            V_store[s] = 0.0

        lowers, uppers = start_bounds[:n], start_bounds[n:]
        for i, var in enumerate(int_vars):
            var.setLower(lowers[i])
            var.setUpper(uppers[i])
            ampl.eval(f"let {var.name}.type := 'continuous';")

        prev_obj = None
        obj_val = None

        for iter_no in range(1, max_iters + 1):
            ampl.solve()
            obj_val = ampl.getObjective(1).value()
            sol = [var.value() for var in int_vars]

            s_prime = normalize_state(sol + sol)
            if s_prime not in V_store:
                V_store[s_prime] = 0.0

            delta_obj = 0.0 if prev_obj is None else (obj_val - prev_obj)
            V_store[s] = delta_obj + V_store[s_prime]

            print(f"Iter {iter_no}: Obj={obj_val:.6f}, ΔObj={delta_obj:.6f}, V(s)={V_store[s]:.6f}")
            prev_obj = obj_val

            # Update bounds
            for i, var in enumerate(int_vars):
                var.setLower(sol[i])
                var.setUpper(sol[i])

            s = s_prime

            if iter_no > 1 and abs(delta_obj) < tol:
                print("Converged.")
                break

        all_results.append({
            "start_vector": start_idx,
            "final_objective": obj_val,
            "final_solution": {v.name: v.value() for v in int_vars},
            "final_state": s,
            "final_V": V_store[s],
            "iterations": iter_no
        })
        ampl.close()

    return all_results, V_store


In [ ]:

# -------------------------
# Example usage
# -------------------------
model_path = "/home/kkishan/Downloads/alan.mod"
data_path  = None

bounds_vectors = generate_valid_vectors(n)
results, V_store = solve_relaxations_with_value_store(model_path, data_path, bounds_vectors, solver="cbc")

print("\nFinal Results Summary:")
for r in results:
    print(f"Start #{r['start_vector']}: Obj={r['final_objective']:.4f}, "
          f"V(s)={r['final_V']:.4f}, Iters={r['iterations']}, State={r['final_state']}")

print("\nStored Value Function Table:")
for s, v in V_store.items():
    print(f"{s}: V={v:.6f}")
